# Evidence Construction

This notebook documents the evidence packet layer for the final pipeline. The final implementation lives in `src/evidence.py`, `src/pipeline.py`, and `scripts/final/refresh_evidence_packets.py`.

The notebook is kept as the narrative/exploratory view; the `.py` script is the reproducible source used to refresh saved report outputs.

In [1]:
import json
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 100)

ROOT = Path('..')
evidence_dir = ROOT / 'reports' / '03_evidence'
summary_path = evidence_dir / 'selected_evidence_summary.csv'
evidence_path = evidence_dir / 'evidence_packets.json'

summary = pd.read_csv(summary_path)
with open(evidence_path) as f:
    evidence_packets = json.load(f)

summary

,case_type,row_position,original_index,y_true,y_pred,y_proba,threshold,prediction_type
0,TP,7773,67105,1,1,0.998875,0.727353,TP
1,FN,7920,35328,1,0,0.019213,0.727353,FN
2,FP,1963,91053,0,1,0.997142,0.727353,FP
3,TN,14835,10107,0,0,0.001510,0.727353,TN


## Final selected cases

The final evidence packets are built for selected TP, FN, FP, and TN cases from the refreshed SHAP analysis. They use the final threshold `0.7274` and the final 379-feature preprocessing schema.

In [2]:
evidence_packets.keys()

dict_keys(['TP', 'FN', 'FP', 'TN'])

In [3]:
tp_packet = evidence_packets['TP']
print(json.dumps(tp_packet, indent=2)[:4000])

{
  "patient_label": "TP",
  "prediction": {
    "y_pred": 1,
    "y_proba": 0.998874831964749,
    "threshold": 0.7273530450209328,
    "y_true": 1,
    "prediction_type": "TP"
  },
  "risk_increasing_evidence": [
    {
      "feature": "d1_heartrate_min",
      "value": 0.0,
      "shap_value": 1.1382217805524988,
      "direction": "risk_increasing",
      "clinical_meaning": "extreme or very low heart rate may indicate severe instability or data quality issue",
      "caution_flags": [
        "Zero-valued vital sign; may reflect extreme clinical event or recording artifact."
      ]
    },
    {
      "feature": "d1_lactate_min",
      "value": 8.7,
      "shap_value": 0.7821194077129602,
      "direction": "risk_increasing",
      "clinical_meaning": "No predefined clinical interpretation available.",
      "caution_flags": []
    },
    {
      "feature": "d1_spo2_min",
      "value": 70.0,
      "shap_value": 0.5613483450223526,
      "direction": "risk_increasing",
      "clin

## Packet structure

Each packet contains prediction metadata and two evidence groups:

- `risk_increasing_evidence`: local SHAP contributions that increase predicted mortality risk
- `risk_decreasing_evidence`: local SHAP contributions that decrease predicted mortality risk

Each feature record includes the observed value, SHAP value, direction, clinical meaning, and caution flags.

In [4]:
def packet_to_table(packet, key):
    return pd.DataFrame(packet[key])

packet_to_table(tp_packet, 'risk_increasing_evidence')

,feature,value,shap_value,direction,clinical_meaning,caution_flags
0,d1_heartrate_min,0.00,1.138222,risk_increasing,extreme or very low heart rate may indicate se...,[Zero-valued vital sign; may reflect extreme c...
1,d1_lactate_min,8.70,0.782119,risk_increasing,No predefined clinical interpretation available.,[]
2,d1_spo2_min,70.00,0.561348,risk_increasing,low minimum oxygen saturation indicates hypoxemia,[]
3,apache_3j_diagnosis,102.01,0.440532,risk_increasing,diagnosis category contributes baseline clinic...,[]
4,gcs_motor_apache,1.00,0.338107,risk_increasing,low GCS motor score indicates impaired neurolo...,[]
5,d1_arterial_ph_min,6.89,0.336470,risk_increasing,No predefined clinical interpretation available.,[]
6,ventilated_apache,1.00,0.325105,risk_increasing,mechanical ventilation indicates severe respir...,[]
7,d1_sysbp_min,60.00,0.314144,risk_increasing,low systolic blood pressure indicates hemodyna...,[]


In [5]:
packet_to_table(tp_packet, 'risk_decreasing_evidence')

,feature,value,shap_value,direction,clinical_meaning,caution_flags
0,ethnicity_African American,1.00,-0.077615,risk_decreasing,No predefined clinical interpretation available.,[]
1,diabetes_mellitus,1.00,-0.064262,risk_decreasing,No predefined clinical interpretation available.,[]
2,h1_lactate_min,8.70,-0.063807,risk_decreasing,No predefined clinical interpretation available.,[]
3,h1_heartrate_max,134.00,-0.048479,risk_decreasing,No predefined clinical interpretation available.,[]
4,h1_arterial_po2_min,62.00,-0.039856,risk_decreasing,No predefined clinical interpretation available.,[]
5,ethnicity_Caucasian,0.00,-0.032690,risk_decreasing,No predefined clinical interpretation available.,[]
6,d1_temp_min,36.27,-0.030779,risk_decreasing,No predefined clinical interpretation available.,[]
7,d1_sodium_max,143.00,-0.026805,risk_decreasing,No predefined clinical interpretation available.,[]


## Caution behavior

The earlier prototype included `icu_id` as a model feature and marked it with a caution flag. In the final preprocessing schema, ID/location columns are removed before modeling.

Current caution flags therefore focus on remaining values that need careful interpretation, especially zero-valued vital signs or unusual timing values when they appear in local evidence.

In [6]:
caution_rows = []
for case_type, packet in evidence_packets.items():
    for section in ['risk_increasing_evidence', 'risk_decreasing_evidence']:
        for record in packet[section]:
            if record.get('caution_flags'):
                caution_rows.append({
                    'case_type': case_type,
                    'section': section,
                    'feature': record['feature'],
                    'value': record['value'],
                    'caution_flags': record['caution_flags'],
                })

pd.DataFrame(caution_rows)

,case_type,section,feature,value,caution_flags
0,TP,risk_increasing_evidence,d1_heartrate_min,0.0,[Zero-valued vital sign; may reflect extreme c...
1,FP,risk_increasing_evidence,d1_heartrate_min,0.0,[Zero-valued vital sign; may reflect extreme c...


## Prompt bridge

The evidence packets are converted into LLM prompts by `src/prompts.py`. The prompt includes model prediction and SHAP evidence, but intentionally excludes `y_true` and TP/FN/FP/TN metadata to prevent label leakage.

In [7]:
prompt_dir = ROOT / 'reports' / '04_llm_reasoning' / 'prompts'
for path in sorted(prompt_dir.glob('*_prompt.txt')):
    print(path.name, '->', len(path.read_text()), 'characters')

fn_prompt.txt -> 5939 characters
fp_prompt.txt -> 6006 characters
tn_prompt.txt -> 5889 characters
tp_prompt.txt -> 6001 characters


## Evidence construction summary

The evidence packet layer turns local SHAP tables into a structured, auditable input for LLM explanation generation. This keeps the LLM grounded in patient-specific model evidence and allows the deterministic validator to check the generated explanation against the same packet.